# CarveFormer — FFT-75 Benchmark (Kaggle Runner)

Full, top-to-bottom benchmark runner for **CarveFormer** (Swin Transformer V2
Tiny + 96-d byte embedding) on the FFT-75 dataset, using the shared **DeepCarv**
framework. This notebook is a benchmark runner, not a demo.

**Switch 512 <-> 4096 by changing `FRAGMENT_SIZE` in the config cell only.**

Steps:
1. Install dependencies
2. Set paths & fragment size
3. Obtain FFT-75
4. Verify the NPZ layout
5. Sanity-train (few steps)
6. Full train
7. Evaluate
8. Save outputs
9. (optional) Compress the run

## 1. Install dependencies

In [ ]:
# timm provides the Swin Transformer V2 Tiny backbone. torch is preinstalled on
# Kaggle GPU images. Keep installs quiet and idempotent.
!pip -q install "timm>=1.0.0" pyyaml >/dev/null 2>&1
import torch, timm
print("torch", torch.__version__, "| timm", timm.__version__, "| cuda", torch.cuda.is_available())

## 2. Set paths and fragment size

This is the **only** cell you edit to switch block size or point at your data.

In [ ]:
import os
from pathlib import Path

# ---- The one switch: 512 or 4096 --------------------------------------------
FRAGMENT_SIZE = 512          # <-- change to 4096 for the 4 KiB benchmark
NUM_CLASSES   = 75           # FFT-75 Scenario #1 (11/25/5/2/2 for #2..#6)
PRETRAINED    = True         # ImageNet1k init (paper default)

# ---- Paths (Kaggle defaults; adjust dataset slug if different) ---------------
REPO_DIR   = Path("/kaggle/working/deepcarv")           # where the repo is cloned/copied
DATA_DIR   = Path("/kaggle/input/fft-75/FFT-75")         # {DATA_DIR}/{fragment}/{train,val,test}.npz
WORK_DIR   = Path("/kaggle/working/carveformer_run")
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("fragment_size", FRAGMENT_SIZE, "| data", DATA_DIR, "| work", WORK_DIR)

## 3. Obtain FFT-75 and the DeepCarv repo

Attach the FFT-75 NPZ dataset as a Kaggle Dataset input, and bring in the
DeepCarv repo (Add Data → your GitHub export, or `git clone`). The benchmark
reads the pre-split `.npz` files directly — **no split regeneration, no CSV.**

In [ ]:
# If the repo is not already present, clone the eval-CarveFormer branch.
# (On Kaggle, add the repo as a Dataset or use an internet-enabled clone.)
import sys
if not REPO_DIR.exists():
    !git clone --branch eval-CarveFormer https://github.com/yuvnahr/deepcarv.git {REPO_DIR} 2>/dev/null || echo "clone skipped (provide repo manually)"

# Make the repo importable (framework `src` + benchmark code).
sys.path.insert(0, str(REPO_DIR))
print("repo on path:", REPO_DIR, "| exists:", REPO_DIR.exists())

## 4. Verify the expected NPZ layout

Confirm `{DATA_DIR}/{FRAGMENT_SIZE}/{train,val,test}.npz` exist and have the
expected arrays before spending GPU time.

In [ ]:
import numpy as np

frag_dir = DATA_DIR / str(FRAGMENT_SIZE)
for split in ("train", "val", "test"):
    p = frag_dir / f"{split}.npz"
    assert p.exists(), f"MISSING: {p}"
    with np.load(p) as d:
        keys = set(d.files)
        # The documented format is uppercase X/y; some loaders expect lowercase.
        xk = "X" if "X" in keys else ("x" if "x" in keys else None)
        yk = "y" if "y" in keys else None
        assert xk and yk, f"{p} keys {keys} — expected X/y"
        X, y = d[xk], d[yk]
        assert X.ndim == 2 and X.shape[1] == FRAGMENT_SIZE, f"{p} X shape {X.shape}"
        assert X.shape[0] == y.shape[0], f"{p} X/y length mismatch"
    print(f"OK  {split}: X={X.shape} y={y.shape} classes~{int(y.max())+1}")
print("NPZ layout verified.")

## 5. Sanity train (a couple of epochs on a subset)

Quick wiring check that config -> dataset -> registry -> Trainer -> Evaluator
all connect, before the full 50-epoch run.

In [ ]:
import yaml
from benchmarks.CarveFormer.scripts.train import run_training

# Load the committed benchmark config and apply notebook overrides.
CONFIG_PATH = REPO_DIR / "benchmarks/CarveFormer/configs/benchmark.yaml"
with open(CONFIG_PATH) as f:
    config = yaml.safe_load(f)

config["dataset"]["fragment_size"] = FRAGMENT_SIZE
config["dataset"]["root_dir"]      = str(DATA_DIR)
config["dataset"]["num_classes"]   = NUM_CLASSES
config["model"]["pretrained"]      = PRETRAINED
config["paths"]["run_outputs"]     = str(WORK_DIR)

# Sanity: 2 epochs only.
sanity = {k: dict(v) if isinstance(v, dict) else v for k, v in config.items()}
sanity["training"] = dict(config["training"]); sanity["training"]["epochs"] = 2
run_training(sanity, overrides={"fragment_size": FRAGMENT_SIZE, "data_dir": str(DATA_DIR)})
print("Sanity pass complete.")

## 6. Full training (paper: 50 epochs, AdamW, lr 3.75e-4, wd 0.05)

The committed config already encodes the paper hyperparameters. Effective batch
size 1024 is documented in the config; adjust `batch_size` to your GPU.

In [ ]:
run_dir = run_training(config, overrides={"fragment_size": FRAGMENT_SIZE, "data_dir": str(DATA_DIR)})
print("Full training complete. Outputs:", run_dir)

## 7. Evaluate the best checkpoint

(The training call above already evaluates on the test split and writes the
standardized outputs; this cell re-runs evaluation standalone if desired.)

In [ ]:
from benchmarks.CarveFormer.scripts.evaluate import run_evaluation

best_ckpt = Path(WORK_DIR) / "checkpoint_best.pt"
if best_ckpt.exists():
    run_evaluation(config, best_ckpt, overrides={"fragment_size": FRAGMENT_SIZE, "data_dir": str(DATA_DIR)})
    print("Standalone evaluation complete.")
else:
    print("No checkpoint found; run training first.")

## 8. Inspect saved outputs

The framework writes the standardized set: `metrics.json`, `summary.json`,
`predictions.csv`, `confusion_matrix.csv`, `per_class_metrics.csv`,
`classification_report.txt`.

In [ ]:
import json
for name in ["summary.json", "metrics.json"]:
    p = Path(WORK_DIR) / name
    if p.exists():
        print("==", name, "==")
        print(json.dumps(json.load(open(p)), indent=2))
print("\nAll files:")
for p in sorted(Path(WORK_DIR).glob("*")):
    print(" ", p.name)

## 9. (Optional) Compress the run for download

In [ ]:
import shutil
archive = shutil.make_archive(f"/kaggle/working/carveformer_fft75_{FRAGMENT_SIZE}", "zip", WORK_DIR)
print("Archive:", archive)